# 08 - Preliminary Subtype Modeling After Expert Review

This notebook continues the project after the completed expert subtype-review export from Supabase. It prepares the reviewed subtype labels for first-pass modeling and summarizes preliminary baselines.

**Important:** these results are exploratory. The Supabase export does not contain original `subject_id`, so these first-pass splits are stratified by label but not yet subject-disjoint.

## Research Position

The broad dermatoglyphic classifier already predicts `arch`, `left_slant_loop`, `right_slant_loop`, and `whorl`. The subtype phase focuses on expert-reviewed generic `arch` and `whorl` records.

The first subtype-modeling pass uses only rows marked `accept` with a specific subtype. Rows marked `unclear`, `adjudicate`, or `exclude` remain outside clean supervised training.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXPORT_DIR = ROOT / "data" / "processed" / "subtype_review_completed_2026-09-06"
RESULTS_BASELINE = ROOT / "results" / "subtype_preliminary_baseline"
RESULTS_TRANSFER = ROOT / "results" / "subtype_preliminary_transfer_baseline"

## Verify Local Export

The export directory is private row-level biometric research data. It is stored under `data/`, which is ignored by Git.

In [ ]:
assert EXPORT_DIR.exists(), EXPORT_DIR

summary_path = EXPORT_DIR / "subtype_review_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))

summary

## Load Modeling Tables

The reviewer-facing PNGs include visible text banners. The modeling tables point to cropped fingerprint-only `320 x 320` inputs created by `scripts/prepare_subtype_modeling_dataset.py`.

In [ ]:
accepted = pd.read_csv(EXPORT_DIR / "accepted_subtype_labels.csv")
first_pass = pd.read_csv(EXPORT_DIR / "subtype_modeling_first_pass_recommended.csv")
arch = pd.read_csv(EXPORT_DIR / "arch_subtype_modeling_dataset.csv")
whorl = pd.read_csv(EXPORT_DIR / "whorl_subtype_modeling_dataset.csv")

len(accepted), len(first_pass), len(arch), len(whorl)

In [ ]:
pd.DataFrame({
    "accepted_all": accepted["confirmed_subtype"].value_counts(),
    "first_pass": first_pass["confirmed_subtype"].value_counts(),
}).fillna(0).astype(int)

## First-Pass Modeling Scope

The subtype work is split into two tasks:

1. Arch subtype: `plain_arch` versus `tented_arch`.
2. Whorl subtype: `plain_whorl`, `central_pocket_loop_whorl`, and `double_loop_whorl`.

`accidental_whorl` has only three accepted examples, so it is excluded from the first-pass model and retained as descriptive evidence.

In [ ]:
prep_summary = json.loads(
    (EXPORT_DIR / "subtype_modeling_summary.json").read_text(encoding="utf-8")
)

prep_summary["subtype_counts_recommended_first_pass"]

## Run Classical Baselines

The classical baseline uses simple deterministic image features: resized grayscale pixels, block-mean intensity summaries, and local gradient-orientation histograms. Row-level predictions are written back under ignored `data/`; aggregate metrics are written under `results/`.

In [ ]:
# Run from the repository root if metrics need to be regenerated.
# !python scripts/train_preliminary_subtype_baselines.py

In [ ]:
classical_metrics = pd.read_csv(
    RESULTS_BASELINE / "subtype_preliminary_baseline_metrics.csv"
)

classical_metrics[["experiment", "model", "accuracy", "balanced_accuracy", "macro_f1"]]

## Run Frozen Transfer-Feature Baselines

The transfer baseline uses frozen ImageNet ResNet-18 embeddings from the cropped fingerprint images, then trains class-balanced linear classifiers. This is still a baseline, not a tuned fingerprint-specific subtype model.

In [ ]:
# Run from the repository root if metrics need to be regenerated.
# !python scripts/train_preliminary_subtype_transfer_baselines.py

In [ ]:
transfer_metrics = pd.read_csv(
    RESULTS_TRANSFER / "subtype_preliminary_transfer_metrics.csv"
)

transfer_metrics[["experiment", "model", "accuracy", "balanced_accuracy", "macro_f1"]]

## Best Preliminary Models

Balanced accuracy and macro F1 are more informative than raw accuracy because the whorl subtype set is strongly imbalanced toward `plain_whorl`.

In [ ]:
all_metrics = pd.concat([classical_metrics, transfer_metrics], ignore_index=True)
best = all_metrics.sort_values(
    ["experiment", "macro_f1"], ascending=[True, False]
).groupby("experiment", as_index=False).head(1)

best[["experiment", "model", "accuracy", "balanced_accuracy", "macro_f1"]]

## Per-Class Review

Per-class recall is essential. A high whorl accuracy can still be weak if the model predicts almost everything as `plain_whorl`.

In [ ]:
whorl_report = pd.read_csv(
    RESULTS_TRANSFER / "whorl_first_pass_classification_report.csv"
)

whorl_report.query("model == 'resnet18_logistic_balanced'")

## Interpretation

The first-pass arch task shows useful separation with the classical class-balanced Linear SVC baseline. The transfer-feature baseline is stronger for the whorl task, especially because it begins to recover minority whorl subtypes instead of predicting only `plain_whorl`.

These results should be treated as engineering evidence, not final research claims. The next defensible step is to join the completed review labels back to the private EfficientNet metadata so every row has `subject_id`, `finger_position`, and `experiment_role`.

## Next Research Step

After joining original metadata, rerun subtype evaluation with grouped subject-disjoint splits. That will align the subtype methodology with the broad-classifier leakage-control strategy already used in this project.

## Metadata Join Checkpoint

The required private metadata file is expected at:

`data/processed/efficientnet_320_package/roll_320_clahe_metadata.csv`

Once restored, join it to the completed subtype labels using `record_key`. This produces the table needed for grouped subject-disjoint evaluation.

In [ ]:
METADATA_PATH = ROOT / "data" / "processed" / "efficientnet_320_package" / "roll_320_clahe_metadata.csv"

METADATA_PATH.exists()

In [ ]:
# Run after the private metadata file is restored.
# !python scripts/join_completed_subtype_labels_to_metadata.py